### Esercizi SQL Esteso

Sono date le seguenti tabele:

**Cella** (*idCella*, città, regione)

**Data** (*idData*, data, mese, semestre, anno)

**Attività** (*idData*, OperatoreVirtuale, idCella, numeroDiTransizioni, minutiTelefonate)

1. Separatamente per ogni città, regione, e anno, calcolare:

- il numero medio giornaliero di transizioni

- la percentuale di transizioni in ogni città rispetto al totale della regione

- all'interno di ogni regione assegnare un rank alle città per numero di transizioni decrescente

In [ ]:
SELECT C.città, C.regione, D.anno, 
    1.0 * SUM(A.numeroDiTransazioni) / COUNT (DISTINCT D.data),
    100 * SUM(A.numeroDiTransazioni) / SUM(SUM(A.numeroDiTransazioni)) OVER(
        PARTITION BY C.regione
    ),
    RANK() OVER(
        PARTITION BY C.regione
        ORDER BY SUM(A.numeroDiTransazioni) DESC
    )
FROM Attività A 
JOIN Cella C ON C.idCella = A.idCella
JOIN Data D ON D.idData = A.idData
GROUP BY C.città, C.regione, D.anno

2. Separatamente per ogni città, mese e operatore virtuale, calcolare:

- il numero totale di minuti di telefonate

- la percentuale di minuti di telefonate di ogni città rispetto al totale mensile dell’operatore su tutte le città

- il cumulativo mensile di minuti di chiamata dall'inizio dell'anno

In [ ]:
SELECT C.città, D.mese, A.OperatoreVirtuale,
    SUM(A.minutiTelefonate),
    100 * SUM(A.minutiTelefonate) / SUM(SUM(A.minutiTelefonate)) OVER (
        PARTITION BY D.mese, A.OperatoreVirtuale
    ),
    SUM(SUM(A.minutiTelefonate)) OVER (
        PARTITION BY C.città, A.OperatoreVirtuale, D.anno
        ORDER BY D.mese ASC
        ROWS UNBOUNDED PRECEDING
    )
FROM Attività A 
JOIN Cella C ON C.idCella = A.idCella
JOIN Data D ON D.idData = A.idData
GROUP BY C.città, D.mese, D.anno, A.OperatoreVirtuale

3. Separatamente per ogni regione, operatore virtuale e mese, calcolare:

- la durata media di una singola transazione in minuti, ovvero minuti totali diviso transizioni totali

- la media mobile su 3 mesi del numero totale di minuti di telefonate, per quella regione e quell'operatore virtuale

- la differenza tra i minuti totali del mese corrente e quelli del mese immediatamente precedente per la stessa regione e lo stesso operatore virtuale

In [ ]:
SELECT C.regione, A.OperatoreVirtuale, D.mese,
    SUM(A.minutiTelefonate) / SUM(A.numeroDiTransazioni),
    AVG(SUM(A.minutiTelefonate)) OVER(
        PARTITION BY C.regione, A.OperatoreVirtuale
        ORDER BY D.mese ASC
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),
    SUM(A.minutiTelefonate) - LAG(SUM(A.minutiTelefonate), 1)OVER(
        PARTITION BY C.regione, A.OperatoreVirtuale
        ORDER BY D.mese
    )

FROM Attività A 
JOIN Cella C ON C.idCella = A.idCella
JOIN Data D ON D.idData = A.idData
GROUP BY C.regione, A.OperatoreVirtuale, D.mese

4. Considerare le seguenti 3 query di interesse frequenti sul Data Warehouse:

    Query 1: Per l'operatore virtuale 'Iliad' (attributo OperatoreVirtuale = 'Iliad'), visualizzare il totale delle transizioni e la media mensile dei minuti di telefonate, separatamente per città e semestre.

    Query 2: Per le celle situate nella regione 'Lombardia', calcolare il numero cumulativo di transizioni al trascorrere dei semestri, separatamente per anno e operatore virtuale.

    Query 3: Separatamente per anno e regione, visualizzare la percentuale dei minuti di telefonate rispetto al totale annuale nazionale.

a) Considerando le informazioni precedenti, definire una vista materializzata (ViewAttività) con **CREATE MATERIALIZED VIEW**, in modo da ridurre il tempo di risposta delle query di interesse sopra riportate. 

In [ ]:
CREATE MATERIALIZED VIEW ViewAttività
BUILD IMMEDIATE
REFRESH FAST ON COMMIT
AS
SELECT D.mese, D.semestre, D.anno, A.OperatoreVirtuale, C.città, C.regione, 
    SUM(A.numeroDiTransazioni) AS TotTransazioni,
    SUM(A.minutiTelefonate) AS TotMinutiTelefonate
FROM Attività A 
JOIN Cella C ON C.idCella = A.idCella
JOIN Data D ON D.idData = A.idData
GROUP BY D.mese, D.semestre, D.anno, A.OperatoreVirtuale, C.città, C.regione ; 

b) Insieme minimale: indicare l'insieme minimale di attributi che identifica univocamente le tuple della vista ViewAttività.

Insieme minimale = {mese, OperatoreVirtuale, città}